In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/ann_customer_churn_50000.csv')

In [3]:
df.head()

,age,tenure_months,monthly_charges,support_tickets,late_payments,usage_hours,satisfaction,contract_monthly,tech_support,auto_payment,churn
0,23,7,76.20,5,1,27.58,9.87,1,1,0,0
1,62,69,53.67,5,1,28.71,6.41,1,0,1,0
2,55,38,86.48,4,1,39.27,4.58,0,1,1,0
3,43,72,79.26,0,1,55.58,7.23,0,1,0,0
4,43,30,69.54,6,1,35.25,5.19,1,0,1,0


In [4]:
df.shape

(50000, 11)

In [5]:
df.isna().sum()

age                 0
tenure_months       0
monthly_charges     0
support_tickets     0
late_payments       0
usage_hours         0
satisfaction        0
contract_monthly    0
tech_support        0
auto_payment        0
churn               0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               50000 non-null  int64  
 1   tenure_months     50000 non-null  int64  
 2   monthly_charges   50000 non-null  float64
 3   support_tickets   50000 non-null  int64  
 4   late_payments     50000 non-null  int64  
 5   usage_hours       50000 non-null  float64
 6   satisfaction      50000 non-null  float64
 7   contract_monthly  50000 non-null  int64  
 8   tech_support      50000 non-null  int64  
 9   auto_payment      50000 non-null  int64  
 10  churn             50000 non-null  int64  
dtypes: float64(3), int64(8)
memory usage: 4.2 MB


<h2>Feature Selection</h2>

In [7]:
X=df.drop(columns=['churn'],axis=1)
y=df['churn']

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
from sklearn.preprocessing import StandardScaler

In [11]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)

In [12]:
X_test_scaled=scaler.transform(X_test)

In [13]:
y_train.to_numpy()

array([0, 0, 0, ..., 0, 0, 0], shape=(40000,))

In [14]:
y_test.to_numpy()

array([0, 0, 1, ..., 0, 0, 0], shape=(10000,))

<h3>Preparing data for pytorchby converting them to tensors</h3>

In [15]:
import torch

In [16]:
X_train_scaled_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy()).reshape(-1, 1)

X_test_scaled_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy()).reshape(-1, 1)

In [17]:
y_train_tensor = y_train_tensor.float()
y_test_tensor = y_test_tensor.float()

In [18]:
print(X_train_scaled_tensor.shape)
print(y_train_tensor.shape)

print(X_test_scaled_tensor.shape)
print(y_test_tensor.shape)

torch.Size([40000, 10])
torch.Size([40000, 1])
torch.Size([10000, 10])
torch.Size([10000, 1])


<h2>Neural Network Model 🧠</h2>

In [19]:
import torch.nn as nn

In [82]:
neural_model=nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    
)

In [83]:
neural_model

Sequential(
  (0): Linear(in_features=10, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=8, bias=True)
  (5): ReLU()
  (6): Linear(in_features=8, out_features=1, bias=True)
)

In [84]:
pos_weight = (y_train_tensor == 0).sum() / (y_train_tensor == 1).sum()

print(pos_weight)

tensor(12.8026)


In [85]:
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [87]:
optimizer = torch.optim.Adam(
    neural_model.parameters(),
    lr=0.001
)

In [91]:
for epoch in range (1,10001):
    optimizer.zero_grad()
    y_pred=neural_model(X_train_scaled_tensor)
    loss=loss_fn(y_pred,y_train_tensor)
    loss.backward()
    optimizer.step()

    print(f'{epoch}: {loss}')

1: 0.7892144322395325
2: 0.7891629338264465
3: 0.789111316204071
4: 0.789059579372406
5: 0.7890079021453857
6: 0.7889552712440491
7: 0.7889017462730408
8: 0.7888477444648743
9: 0.7887943387031555
10: 0.7887415289878845
11: 0.788689374923706
12: 0.788637638092041
13: 0.7885869145393372
14: 0.7885364294052124
15: 0.7884861826896667
16: 0.788437008857727
17: 0.7883878350257874
18: 0.7883386015892029
19: 0.78829026222229
20: 0.788242757320404
21: 0.7881959080696106
22: 0.7881490588188171
23: 0.7881012558937073
24: 0.7880533337593079
25: 0.7880055904388428
26: 0.7879582047462463
27: 0.7879108786582947
28: 0.7878627777099609
29: 0.7878141403198242
30: 0.7877643704414368
31: 0.787713885307312
32: 0.7876614332199097
33: 0.7876086831092834
34: 0.7875558733940125
35: 0.7875028252601624
36: 0.7874498963356018
37: 0.7873976826667786
38: 0.7873459458351135
39: 0.7872948050498962
40: 0.7872433662414551
41: 0.7871924638748169
42: 0.7871408462524414
43: 0.7870893478393555
44: 0.7870385050773621
45: 0.

In [54]:
y_test_pred = neural_model(X_test_scaled_tensor)

In [55]:
y_test_pred.shape

torch.Size([10000, 1])

In [56]:
loss_fn(y_test_pred,y_test_tensor)

tensor(0.1888, grad_fn=<BinaryCrossEntropyBackward0>)

In [57]:
y_test_pred_classes=(y_test_pred>=0.5).float()

In [58]:
y_test_pred_classes[:10]

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])

In [59]:
torch.unique(y_test_pred_classes,return_counts=True)

(tensor([0., 1.]), tensor([9766,  234]))

In [60]:
print(y_test_pred.min())
print(y_test_pred.max())

tensor(1.2173e-06, grad_fn=<MinBackward1>)
tensor(0.8154, grad_fn=<MaxBackward1>)


In [61]:
print(torch.unique(y_test_tensor, return_counts=True))

(tensor([0., 1.]), tensor([9275,  725]))


In [42]:
print(y_test_pred[y_test_tensor == 1].mean())
print(y_test_pred[y_test_tensor == 0].mean())

tensor(0.2657, grad_fn=<MeanBackward0>)
tensor(0.2589, grad_fn=<MeanBackward0>)


In [43]:
from sklearn.metrics import classification_report
print(classification_report(y_test_tensor.numpy(), y_test_pred_classes.numpy()))

              precision    recall  f1-score   support

         0.0       0.93      1.00      0.96      9275
         1.0       0.00      0.00      0.00       725

    accuracy                           0.93     10000
   macro avg       0.46      0.50      0.48     10000
weighted avg       0.86      0.93      0.89     10000



c:\Users\Kartik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Kartik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Kartik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [44]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_classes.numpy()
)

print(cm)

[[9275    0]
 [ 725    0]]


In [45]:
print(y_test_pred.min())
print(y_test_pred.max())
print(y_test_pred.mean())

tensor(0.1099, grad_fn=<MinBackward1>)
tensor(0.3867, grad_fn=<MaxBackward1>)
tensor(0.2594, grad_fn=<MeanBackward0>)


In [46]:
y_test_pred_class_03 = (y_test_pred >= 0.3).float()

print(torch.unique(y_test_pred_class_03, return_counts=True))

(tensor([0., 1.]), tensor([8487, 1513]))


In [47]:
cm_03 = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class_03.numpy()
)

print(cm_03)

[[7908 1367]
 [ 579  146]]


In [48]:
print("Actual churners:", y_test_pred[y_test_tensor == 1].mean())
print("Actual non-churners:", y_test_pred[y_test_tensor == 0].mean())

Actual churners: tensor(0.2657, grad_fn=<MeanBackward0>)
Actual non-churners: tensor(0.2589, grad_fn=<MeanBackward0>)


In [50]:
test_loss = loss_fn(y_test_pred, y_test_tensor)

print("Train loss:", loss.item())
print("Test loss:", test_loss.item())

Train loss: 0.378553569316864
Test loss: 0.3761949837207794


In [62]:
y_test_pred = neural_model(X_test_scaled_tensor)

y_test_pred_class = (y_test_pred >= 0.5).float()

cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(cm)

[[9202   73]
 [ 564  161]]


In [63]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(accuracy)

0.9363


In [73]:
y_test_pred = neural_model(X_test_scaled_tensor)
y_test_pred_class = (y_test_pred >= 0.5).float()

cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(cm)

[[9177   98]
 [ 541  184]]


In [89]:
y_test_logits = neural_model(X_test_scaled_tensor)

y_test_pred = torch.sigmoid(y_test_logits)

y_test_pred_class = (y_test_pred >= 0.5).float()

In [90]:
cm = confusion_matrix(
    y_test_tensor.numpy(),
    y_test_pred_class.numpy()
)

print(cm)

[[7300 1975]
 [ 156  569]]
